In [1]:
import os
import shutil
import random
from pathlib import Path

# Path where your current dataset is stored (with 10 subfolders)
SOURCE_DIR = "C:/Users/srkap/OneDrive/Desktop/skin/IMG_CLASSES"        # change this to your dataset root
TARGET_DIR = "dataset"             # new folder will be created
TRAIN_RATIO = 0.8                  # 80% train, 20% test
SEED = 42

random.seed(SEED)

# Make target structure
train_path = Path(TARGET_DIR, "train")
test_path = Path(TARGET_DIR, "test")
train_path.mkdir(parents=True, exist_ok=True)
test_path.mkdir(parents=True, exist_ok=True)

# Get all classes (subfolders)
classes = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))]
print("Found classes:", classes)

for cls in classes:
    src_cls = Path(SOURCE_DIR, cls)
    train_cls = Path(train_path, cls)
    test_cls = Path(test_path, cls)
    train_cls.mkdir(parents=True, exist_ok=True)
    test_cls.mkdir(parents=True, exist_ok=True)

    files = [f for f in os.listdir(src_cls) if os.path.isfile(os.path.join(src_cls, f))]
    random.shuffle(files)

    split_idx = int(len(files) * TRAIN_RATIO)
    train_files = files[:split_idx]
    test_files = files[split_idx:]

    for f in train_files:
        shutil.copy2(src_cls / f, train_cls / f)
    for f in test_files:
        shutil.copy2(src_cls / f, test_cls / f)

    print(f"Class '{cls}': {len(train_files)} train, {len(test_files)} test")

print("✅ Dataset successfully split into train/ and test/ folders.")


Found classes: ['1. Eczema', '10. Warts Molluscum and other Viral Infections', '2. Melanoma', '3. Atopic Dermatitis', '4. Basal Cell Carcinoma (BCC)', '5. Melanocytic Nevi (NV)', '6. Benign Keratosis-like Lesions (BKL)', '7. Psoriasis pictures Lichen Planus and related diseases', '8. Seborrheic Keratoses and other Benign Tumors', '9. Tinea Ringworm Candidiasis and other Fungal Infections']
Class '1. Eczema': 864 train, 216 test
Class '10. Warts Molluscum and other Viral Infections': 1682 train, 421 test
Class '2. Melanoma': 756 train, 189 test
Class '3. Atopic Dermatitis': 948 train, 237 test
Class '4. Basal Cell Carcinoma (BCC)': 1603 train, 401 test
Class '5. Melanocytic Nevi (NV)': 1238 train, 310 test
Class '6. Benign Keratosis-like Lesions (BKL)': 796 train, 200 test
Class '7. Psoriasis pictures Lichen Planus and related diseases': 960 train, 240 test
Class '8. Seborrheic Keratoses and other Benign Tumors': 885 train, 222 test
Class '9. Tinea Ringworm Candidiasis and other Fungal 

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

# =======================
# CONFIG
# =======================
DATASET_DIR = "C:/Users/srkap/OneDrive/Desktop/skin/dataset"  # where train/ and test/ are stored
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 3
NUM_CLASSES = 10
AUTOTUNE = tf.data.AUTOTUNE
MODEL_SAVE = "skin_cnn_transformer.h5"
# =======================

# --- Load datasets ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "train"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "test"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)

class_names = train_ds.class_names
print("Classes:", class_names)

# --- Data augmentation and normalization ---
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

normalization_layer = layers.Rescaling(1./255)

def prepare(ds, shuffle=False, augment=False):
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.map(lambda x, y: (tf.image.resize(x, (IMG_SIZE, IMG_SIZE)), y), num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare(train_ds, shuffle=True, augment=True)
test_ds = prepare(test_ds)

# --- Hybrid CNN + Transformer model ---
def build_model(img_size=IMG_SIZE, num_classes=NUM_CLASSES, transformer_units=128, num_heads=4, ff_dim=128, num_transformer_blocks=2):
    inputs = keras.Input(shape=(img_size, img_size, 3))

    # CNN backbone (EfficientNetB0)
    backbone = tf.keras.applications.EfficientNetB0(include_top=False, input_tensor=inputs, weights="imagenet")
    backbone.trainable = False  # freeze first, then unfreeze for fine-tuning
    x = backbone.output

    # Reduce channels
    x = layers.Conv2D(64, kernel_size=1, activation="relu")(x)

    # Flatten spatial -> sequence
    h, w, c = x.shape[1], x.shape[2], x.shape[3]
    seq_len = h * w
    x = layers.Reshape((seq_len, c))(x)

    # Positional embedding
    pos_emb = layers.Embedding(input_dim=seq_len, output_dim=c)
    positions = tf.range(start=0, limit=seq_len, delta=1)
    x = x + pos_emb(positions)

    # Transformer encoder blocks
    for _ in range(num_transformer_blocks):
        # Layer norm + Multi-Head Attention
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=c//num_heads)(x1, x1)
        x = layers.Add()([x, attn])

        # Feed-forward
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        ff = layers.Dense(ff_dim, activation="relu")(x1)
        ff = layers.Dense(c)(ff)
        x = layers.Add()([x, ff])

    # Classification head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(transformer_units, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)

model = build_model()
model.summary()

# --- Compile ---
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

# --- Callbacks ---
callbacks = [
    keras.callbacks.ModelCheckpoint(MODEL_SAVE, save_best_only=True, monitor="val_accuracy", mode="max"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

# --- Train ---
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# --- Fine-tune backbone (optional) ---
print("Fine-tuning EfficientNet backbone...")
model.layers[1].trainable = True  # unfreeze EfficientNet
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=test_ds, epochs=5, callbacks=callbacks)

# --- Save final model in multiple formats ---
model.save("final_skin_cnn_transformer.keras")   # ✅ recommended new format
model.save("final_skin_cnn_transformer.h5")      # ✅ legacy HDF5 format
model.export("final_skin_cnn_transformer")       # ✅ TensorFlow SavedModel format

print("✅ Training complete. Model saved as .keras, .h5 and TensorFlow SavedModel!")


Found 10588 files belonging to 10 classes.
Found 2651 files belonging to 10 classes.
Classes: ['1. Eczema', '10. Warts Molluscum and other Viral Infections', '2. Melanoma', '3. Atopic Dermatitis', '4. Basal Cell Carcinoma (BCC)', '5. Melanocytic Nevi (NV)', '6. Benign Keratosis-like Lesions (BKL)', '7. Psoriasis pictures Lichen Planus and related diseases', '8. Seborrheic Keratoses and other Benign Tumors', '9. Tinea Ringworm Candidiasis and other Fungal Infections']


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling_1[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_2[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,208,109 (16.05 MB)

 Trainable params: 158,538 (619.29 KB)

...

331/331 ━━━━━━━━━━━━━━━━━━━━ 389s 943ms/step - accuracy: 0.1255 - loss: 2.3672 - val_accuracy: 0.1588 - val_loss: 2.2627 - learning_rate: 3.0000e-04
Epoch 2/3
331/331 ━━━━━━━━━━━━━━━━━━━━ 361s 1s/step - accuracy: 0.1515 - loss: 2.2871 - val_accuracy: 0.1588 - val_loss: 2.2632 - learning_rate: 3.0000e-04
Epoch 3/3
331/331 ━━━━━━━━━━━━━━━━━━━━ 337s 944ms/step - accuracy: 0.1561 - loss: 2.2783 - val_accuracy: 0.1513 - val_loss: 2.2668 - learning_rate: 3.0000e-04
Fine-tuning EfficientNet backbone...
Epoch 1/5
331/331 ━━━━━━━━━━━━━━━━━━━━ 403s 1s/step - accuracy: 0.1546 - loss: 2.2725 - val_accuracy: 0.1513 - val_loss: 2.2628 - learning_rate: 1.0000e-05
Epoch 2/5
331/331 ━━━━━━━━━━━━━━━━━━━━ 402s 1s/step - accuracy: 0.1477 - loss: 2.2757 - val_accuracy: 0.1513 - val_loss: 2.2627 - learning_rate: 1.0000e-05
Epoch 3/5
331/331 ━━━━━━━━━━━━━━━━━━━━ 471s 1s/step - accuracy: 0.1506 - loss: 2.2741 - val_accuracy: 0.1588 - val_loss: 2.2624 - learning_rate: 1.0000e-05
Epoch 4/5
331/331 ━━━━━━━━━━━━━

331/331 ━━━━━━━━━━━━━━━━━━━━ 430s 1s/step - accuracy: 0.1493 - loss: 2.2773 - val_accuracy: 0.1713 - val_loss: 2.2621 - learning_rate: 1.0000e-05
Epoch 5/5
331/331 ━━━━━━━━━━━━━━━━━━━━ 393s 1s/step - accuracy: 0.1517 - loss: 2.2693 - val_accuracy: 0.1513 - val_loss: 2.2634 - learning_rate: 1.0000e-05


INFO:tensorflow:Assets written to: final_skin_cnn_transformer\assets


INFO:tensorflow:Assets written to: final_skin_cnn_transformer\assets


Saved artifact at 'final_skin_cnn_transformer'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_5')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2608489037888: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  2608489038416: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  2608449516672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2608449515088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2608449517552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2608449505584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2608449516496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2608449517376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2608450193360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2608450196176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  260